In [1]:
################
# Tabular ARGN #
################

## Authors:             Ricardo Zambrano
## email:               rzambrano@gmail.com
## Date:               2026-03-14

## reference:           "TabularARGN: A flexible and Efficient Auto-Regressive Framework for Generating high-Fidelity Synthetic Data"
##                      Tiwald et al. (2025)
##                      https://arxiv.org/pdf/2501.12012

######################
# Tabular ARGN       #
######################

## Author:             Ricardo Zambrano
## Email:              rzambrano@gmail.com
## Date:               2026-03-14
## Reference:          "TabularARGN: A Flexible and Efficient Auto-Regressive Framework
##                      for Generating High-Fidelity Synthetic Data"
##                      Tiwald et al. (2025)
##                      https://arxiv.org/abs/[arxiv-id]  # fill this in

#################
# Session Setup #
#################

# ── Standard Library ──────────────────────────────────────────────────────────
import os
import random
from pathlib import Path

from typing import Protocol

from datetime import date, datetime

# ── Numerical & Data ──────────────────────────────────────────────────────────
import numpy as np
import polars as pl
import pandas as pd

# ── Visualization ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

# ── PyTorch ───────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ── Experiment Tracking ───────────────────────────────────────────────────────
import wandb
import mlflow
from omegaconf import DictConfig, OmegaConf
import hydra

# ── Evaluation ────────────────────────────────────────────────────────────────
from sklearn.metrics import f1_score, roc_auc_score
import optuna

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

# ── Hardcoded Variables ────────────────────────────────────────────────────────

# -- None --


c:\Users\rzamb\Documents\tabular_nn\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ── Working Directory ──────────────────────────────────────────────────────────

current_dir = Path.cwd().resolve()
print(current_dir)

target_dir = Path(r"c:\Users\rzamb\Documents\tabular_nn").resolve()

if current_dir != target_dir:

    # Get the path object for two levels up
    two_levels_up = current_dir.parents[0]

    # Change the working directory to the new path
    os.chdir(two_levels_up)

# Optional: Print new working directory to confirm the change
print(f"New working directory: {Path.cwd()}")

C:\Users\rzamb\Documents\tabular_nn\notebooks
New working directory: C:\Users\rzamb\Documents\tabular_nn


In [3]:
####################
# Helper Functions #
####################

# -- None --

In [4]:
####################
# Data Loading     #
####################

insurance_data = pd.read_csv("./data/raw/insurance/insurance.csv")

In [5]:
insurance_data.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [6]:
insurance_data.describe()

,age,bmi,children,charges
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.663397,1.094918,13270.422265
std,14.049960,6.098187,1.205493,12110.011237
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.296250,0.000000,4740.287150
50%,39.000000,30.400000,1.000000,9382.033000
75%,51.000000,34.693750,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


In [7]:
insurance_data.dtypes

age           int64
sex          object
bmi         float64
children      int64
smoker       object
region       object
charges     float64
dtype: object

In [8]:
beijing_data = pd.read_csv("./data/raw/beijing/PRSA_Data_Dingling_20130301-20170228.csv")

In [9]:
beijing_data.head()

,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station
0,1,2013,3,1,0,4.0,4.0,3.0,NaN,200.0,82.0,-2.3,1020.8,-19.7,0.0,E,0.5,Dingling
1,2,2013,3,1,1,7.0,7.0,3.0,NaN,200.0,80.0,-2.5,1021.3,-19.0,0.0,ENE,0.7,Dingling
2,3,2013,3,1,2,5.0,5.0,3.0,2.0,200.0,79.0,-3.0,1021.3,-19.9,0.0,ENE,0.2,Dingling
3,4,2013,3,1,3,6.0,6.0,3.0,NaN,200.0,79.0,-3.6,1021.8,-19.1,0.0,NNE,1.0,Dingling
4,5,2013,3,1,4,5.0,5.0,3.0,NaN,200.0,81.0,-3.5,1022.3,-19.4,0.0,N,2.1,Dingling


In [10]:
beijing_data["sample_date"] = pd.to_datetime({
    'year': beijing_data['year'],
    'month': beijing_data['month'],
    'day': beijing_data['day']
})

In [11]:
beijing_data = beijing_data.drop(columns=["year", "month", "day"])

In [12]:
beijing_data.dtypes

No                      int64
hour                    int64
PM2.5                 float64
PM10                  float64
SO2                   float64
NO2                   float64
CO                    float64
O3                    float64
TEMP                  float64
PRES                  float64
DEWP                  float64
RAIN                  float64
wd                     object
WSPM                  float64
station                object
sample_date    datetime64[ns]
dtype: object

In [13]:
####################
# Data Wrangling   #
####################

In [14]:
# Introducing a null value

insurance_data.iloc[0,0] = pd.NA
insurance_data.iloc[1,0] = np.nan
insurance_data.iloc[2,1] = None

In [15]:

df_pl = pl.from_pandas(insurance_data).clone()

In [16]:
df_pl.head()

age,sex,bmi,children,smoker,region,charges
f64,str,f64,i64,str,str,f64
null,"""female""",27.9,0,"""yes""","""southwest""",16884.924
null,"""male""",33.77,1,"""no""","""southeast""",1725.5523
28.0,null,33.0,3,"""no""","""southeast""",4449.462
33.0,"""male""",22.705,0,"""no""","""northwest""",21984.47061
32.0,"""male""",28.88,0,"""no""","""northwest""",3866.8552


In [17]:
nrow = df_pl.height

In [18]:
df_pl.null_count()

age,sex,bmi,children,smoker,region,charges
u32,u32,u32,u32,u32,u32,u32
2,1,0,0,0,0,0


In [19]:
col_names = df_pl.columns

In [20]:
col_names

['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges']

In [21]:
df_dtypes = [str(df_pl[name].dtype) for name in col_names]

In [22]:
df_dtypes

['Float64', 'String', 'Float64', 'Int64', 'String', 'String', 'Float64']

In [23]:
print(type(df_dtypes[1]))

<class 'str'>


In [24]:
for i,dtype in enumerate(df_dtypes):
    if dtype in ["String", "Categorical", "Categories", "Enum", "Utf8"]:
        print(i,dtype)

1 String
4 String
5 String


In [25]:
df_pl[:,1].unique().to_list()

['female', 'male', None]

In [32]:
categorical_encode_maps = {}

for i,dtype in enumerate(df_dtypes):
    if dtype in ["String", "Categorical", "Categories", "Enum", "Utf8"]:
        print(i,col_names[i],df_pl[:,i].unique().to_list())

        unique_vals = df_pl[:, i].unique().to_list()
        col = col_names[i]

        if len(unique_vals) > nrow/3:
            Warning(f"Check {col} does not contain open ended values. This implementation only process categorical levels...")

        map_name = col + "_map"

        categorical_encode_maps[map_name] = {
            val: idx for idx, val in enumerate(unique_vals)
        }


1 sex ['male', 'female', None]
4 smoker ['yes', 'no']
5 region ['southwest', 'northwest', 'northeast', 'southeast']


In [33]:
categorical_encode_maps

{'sex_map': {'male': 0, 'female': 1, None: 2},
 'smoker_map': {'no': 0, 'yes': 1},
 'region_map': {'southeast': 0,
  'northwest': 1,
  'northeast': 2,
  'southwest': 3}}

In [34]:
len(categorical_encode_maps)

3

In [52]:
def encode_strings(df_dtypes:list[str], col_names:list[str], df_pl:pl.DataFrame, nrow:int) -> dict[dict[str,int]]:
    """
    Creates a mapping for categorical variables. Assumes columns with string data types are categorical variables
    coded with levels as strings.

    Note the missing values (null) will be casted as None 

    Parameters:
    ----------

    df_dtypes : list[str]
        A list of data types of each column in df_pl
    col_names : list[str]
        A list of column names in df_pl
    df_pl : pl.DataFrame
        A polars data frame
    nrow : int
        Number of rows in df_pl

    Returns:
    -------

    categorical_encode_maps : dict[dict[str,int]]
        A dictionary with dictionaries that map uniques levels to integers
    categorical_columns : list[str]
        A list of columns with categorical strings

    Warns:
    -----

    Warning:
        If the number of levels in a given column are more than a third of the number of 
        rows, the client receives a warning to make sure a column with open ended text 
        was not passed to the model.

    """
    
    categorical_encode_maps = {}
    categorical_columns = []

    for i,dtype in enumerate(df_dtypes):
        if dtype in ["String", "Categorical", "Categories", "Enum", "Utf8"]:

            unique_vals = df_pl[:, i].unique().to_list()
            col = col_names[i]
            
            categorical_columns.append(col)
            if len(unique_vals) > nrow/3:
                Warning(f"Check {col} does not contain open ended values. This implementation only process categorical levels...")

            map_name = col

            categorical_encode_maps[map_name] = {
                val: idx for idx, val in enumerate(unique_vals)
            }

    return categorical_encode_maps, categorical_columns

In [53]:
categorical_encode_maps, catgorical_cols = encode_strings(df_dtypes, col_names, df_pl, nrow)

In [54]:
categorical_encode_maps

{'sex': {None: 0, 'female': 1, 'male': 2},
 'smoker': {'no': 0, 'yes': 1},
 'region': {'northwest': 0, 'southwest': 1, 'northeast': 2, 'southeast': 3}}

In [55]:
catgorical_cols

['sex', 'smoker', 'region']

In [56]:
categorical_decode_maps = {
    outer_key: {v: k for k, v in inner_dict.items()}
    for outer_key, inner_dict in categorical_encode_maps.items()
}

In [57]:
categorical_decode_maps

{'sex': {0: None, 1: 'female', 2: 'male'},
 'smoker': {0: 'no', 1: 'yes'},
 'region': {0: 'northwest', 1: 'southwest', 2: 'northeast', 3: 'southeast'}}

In [58]:
df_pl.head(15)

age,sex,bmi,children,smoker,region,charges
f64,str,f64,i64,str,str,f64
null,"""female""",27.9,0,"""yes""","""southwest""",16884.924
null,"""male""",33.77,1,"""no""","""southeast""",1725.5523
28.0,null,33.0,3,"""no""","""southeast""",4449.462
33.0,"""male""",22.705,0,"""no""","""northwest""",21984.47061
32.0,"""male""",28.88,0,"""no""","""northwest""",3866.8552
…,…,…,…,…,…,…
25.0,"""male""",26.22,0,"""no""","""northeast""",2721.3208
62.0,"""female""",26.29,0,"""yes""","""southeast""",27808.7251
23.0,"""male""",34.4,0,"""no""","""southwest""",1826.843


In [64]:
def encode_categorical(df_pl:pl.DataFrame, encode_map:dict[dict[str,int]],cat_cols:list[str]):
    """
    Assumes a polars data frame and transform the categorical columns into integer levels.

    Parameters:
    ----------

    df_pl : pl.DataFrame
        A polars data frame with categorical data encoded as strings
    
    encode_map : dict[dict[str,int]] 
        A dict of dicts with encoding mapping of string levels into integer levels
    
    cat_cols : list[str]
        List of columns to encode

    Returns:
    -------

    df_pl_encoded : pl.DataFrame
        A polars data frame with categorical variables encoded as integers

    """

    df_pl_encoded = df_pl

    for col_name in cat_cols:
        df_pl_encoded = df_pl_encoded.with_columns(
            pl.col(col_name)
            .replace(encode_map[col_name])
            .cast(pl.Int64)
        )

    return df_pl_encoded

In [67]:
df_pl_t1 = encode_categorical(df_pl, categorical_encode_maps, catgorical_cols)

In [68]:
df_pl_t1.head()

age,sex,bmi,children,smoker,region,charges
f64,i64,f64,i64,i64,i64,f64
null,1,27.9,0,1,1,16884.924
null,2,33.77,1,0,3,1725.5523
28.0,0,33.0,3,0,3,4449.462
33.0,2,22.705,0,0,0,21984.47061
32.0,2,28.88,0,0,0,3866.8552
